In [1]:
from pathlib import Path
import sys

PROJECT_ROOT = Path(
    "/home/Drek/multi-agent-llm-system"
).resolve()

# Помещаем корень проекта в начало sys.path
if str(PROJECT_ROOT) in sys.path:
    sys.path.remove(str(PROJECT_ROOT))

sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)

Project root: /home/Drek/multi-agent-llm-system


In [2]:
# Корень проекта уже задан в первой ячейке.
assert str(PROJECT_ROOT) in sys.path
print("Project root:", PROJECT_ROOT)

Project root: /home/Drek/multi-agent-llm-system


In [3]:
import importlib
import src.llm.client as llm_client_module

importlib.reload(llm_client_module)

LLMClient = llm_client_module.LLMClient
LLMClientError = llm_client_module.LLMClientError
LLMAPIError = llm_client_module.LLMAPIError

client = LLMClient(timeout=30.0)  # тот же timeout, что в прямом примере

try:
    response = client.generate(
        model="nvidia/nemotron-3-ultra-550b-a55b:free",
        messages=[
            {"role": "user", "content": "Ты знаешь английский?"}
        ],
        max_tokens=100,
    )
    print(response.content)
except LLMClientError as exc:
    print(type(exc).__name__, str(exc))
    if isinstance(exc, LLMAPIError):
        print("HTTP status:", exc.status_code)
        print("Request ID:", exc.request_id)
finally:
    client.close()

RateLimitError OpenRouter rate limit was exceeded.


In [4]:
from src.llm import LLMClient, ModelRegistry

# 1. Загружаем реестр моделей из config/models.yaml
registry = ModelRegistry()

# 2. Получаем конфигурацию модели по логическому ID
model = registry.get_model("reasoning")

print("Logical ID: reasoning")
print("Actual model ID:", model.model_id)

# 3. Отправляем запрос через собственный LLMClient
with LLMClient() as client:
    response = client.generate(
        model=model.model_id,
        messages=[
            {
                "role": "user",
                "content": "Объясни простыми словами, что такое Multi-Agent System."
            }
        ],
        max_tokens=300,
    )

print("\nModel response:")
print(response.content)

print("\nToken usage:")
print("Prompt tokens:", response.prompt_tokens)
print("Completion tokens:", response.completion_tokens)
print("Total tokens:", response.total_tokens)

Logical ID: reasoning
Actual model ID: nvidia/nemotron-3-ultra-550b-a55b:free


RateLimitError: OpenRouter rate limit was exceeded.

In [5]:
from src.agents import AgentConfigRegistry
from src.llm import LLMClient

# Загружаем конфигурацию агента
agent = AgentConfigRegistry().get_agent("coding")

print("Agent: coding")
print("Model:", agent.model_id)
print("Generation parameters:", agent.generation_params)

# Отправляем реальный запрос
with LLMClient() as client:
    response = client.generate(
        model=agent.model_id,
        messages=[
            {
                "role": "user",
                "content": "Напиши Python-функцию, которая проверяет, является ли число простым."
            }
        ],
        **agent.generation_params,
    )

print("\nResponse:")
print(response.content)

print("\nFinish reason:", response.finish_reason)
print("Total tokens:", response.total_tokens)

Agent: coding
Model: cohere/north-mini-code:free
Generation parameters: {'temperature': 0.05, 'max_tokens': 3000}


RateLimitError: OpenRouter rate limit was exceeded.

In [ ]:
from src.agents import BaseAgent, AgentConfigRegistry, AgentResult
from src.llm import LLMClient

class NotebookAgent(BaseAgent):
    pass


agent_config = AgentConfigRegistry().get_agent("coding")

client = LLMClient(timeout=30.0)

agent = NotebookAgent(
    agent_id="coding-notebook-1",
    name="Notebook Coding Agent",
    system_prompt=(
        "Ты помощник по Python-программированию. "
        "Отвечай на русском языке, кратко и точно."
    ),
    client=client,
    config=agent_config,
)

print("Agent created successfully!")
print("Agent ID:", agent.agent_id)
print("Agent name:", agent.name)
print("Model:", agent_config.model_id)

Agent created successfully!
Agent ID: coding-notebook-1
Agent name: Notebook Coding Agent
Model: cohere/north-mini-code:free


In [ ]:
try:
    result: AgentResult = agent.run(
        "Напиши функцию Python, которая складывает два числа."
    )

    print("Success:", result.success)
    print("Model:", result.model)

    if result.success:
        print("\nResponse:")
        print(result.content)
    else:
        print("\nError:", result.error)

    print("\nPrompt tokens:", result.prompt_tokens)
    print("Completion tokens:", result.completion_tokens)

finally:
    client.close()

Success: True
Model: cohere/north-mini-code:free

Response:
```python
def add(a: float, b: float) -> float:
    """
    Возвращает сумму двух чисел.

    Параметры:
        a (float): Первое слагаемое.
        b (float): Второе слагаемое.

    Возвращает:
        float: Сумма a и b.
    """
    return a + b
```

Prompt tokens: 36
Completion tokens: 114


In [ ]:
import json

from src.agents import AgentConfigRegistry, BaseAgent
from src.llm import LLMClient


class LoggingCheckAgent(BaseAgent):
    pass


agent_config = AgentConfigRegistry().get_agent("coding")
agent_id = "notebook-logging-check"

with LLMClient(timeout=30.0) as client:
    agent = LoggingCheckAgent(
        agent_id=agent_id,
        name="Logging Check Agent",
        system_prompt="Отвечай кратко и по-русски.",
        client=client,
        config=agent_config,
    )
    result = agent.run("Ответь одним коротким предложением: что такое JSONL?")

print("Успех:", result.success)
print("Модель:", result.model)
print("Ответ:", result.content if result.success else result.error)

log_file = PROJECT_ROOT / "logs" / "llm_requests.jsonl"
log_lines = log_file.read_text(encoding="utf-8").splitlines()
last_log = json.loads(log_lines[-1])

print("\nПоследняя запись лога:")
print("agent_id:", last_log["agent_id"])
print("model_id:", last_log["model_id"])
print("duration_ms:", last_log["duration_ms"])
print("success:", last_log["success"])
print("prompt_tokens:", last_log["prompt_tokens"])
print("completion_tokens:", last_log["completion_tokens"])
print("total_tokens:", last_log["total_tokens"])

Успех: True
Модель: cohere/north-mini-code:free
Ответ: JSONL (JSON Lines) — это текстовый формат данных, в котором каждая строка файла представляет собой отдельный, автономный JSON-объект.

Последняя запись лога:
agent_id: notebook-logging-check
model_id: cohere/north-mini-code:free
duration_ms: 2456.717
success: True
prompt_tokens: 26
completion_tokens: 84
total_tokens: 110


In [ ]:
from src.agents import AgentConfigRegistry, CodingAgent
from src.llm import LLMClient

coding_agent = CodingAgent(
    client=LLMClient(timeout=30.0),
    config=AgentConfigRegistry().get_agent("coding"),
    agent_id="coding-notebook-1",
)

result = coding_agent.run(
    "Напиши функцию Python is_palindrome(text), которая проверяет, является ли строка палиндромом, "
    "игнорируя регистр и пробелы. Верни тесты и инструкции по запуску."
)

print("Успешно:", result.success)

if not result.success:
    print("Ошибка:", result.error)
else:
    solution = coding_agent.parse_solution(result)
    print("Summary:\n", solution.summary)
    print("Code:\n", solution.code)
    print("Tests:\n", solution.tests)
    print("Dependencies:", solution.dependencies)
    print("Run instructions:\n", solution.run_instructions)

Успешно: True
Summary:
 Реализована функция is_palindrome(text), которая проверяет, является ли строка палиндромом, игнорируя регистр и пробелы. Для этого удаляются все пробельные символы и выполняется сравнение строки с её обратным вариантом. Включены тесты с использованием модуля unittest, охватывающие простые палиндромы, строки, состоящие только из пробелов, пустые строки, и не-палиндромы. Код обрабатывает граничные случаи и соответствует спецификации.
Code:
 def is_palindrome(text: str) -> bool:
    # Удаляем все пробельные символы и приводим к нижнему регистру
    cleaned = ''.join(text.split()).lower()
    # Сравниваем строку с её обратным вариантом
    return cleaned == cleaned[::-1]

import unittest

class TestIsPalindrome(unittest.TestCase):
    def test_simple_palindrome(self):
        self.assertTrue(is_palindrome("Racecar"))
        self.assertTrue(is_palindrome("racecar"))
        self.assertTrue(is_palindrome("A man a plan a canal Panama"))
        self.assertTrue(is_pali

In [ ]:
result = coding_agent.run(
    "Доработай функцию палиндрома: игнорируй регистр, пробелы и знаки препинания.",
    additional_context="Сохрани простую реализацию без внешних библиотек.",
    existing_code="def is_palindrome(text):\n    return text == text[::-1]",
    test_requirements="Проверь пустую строку, фразу с пробелами и знаками препинания.",
)

print("Успешно:", result.success)

if result.success:
    solution = coding_agent.parse_solution(result)
    print(solution.code)
    print(solution.tests)
else:
    print("Ошибка:", result.error)

Успешно: True
def is_palindrome(text):
    # Normalize: lower case and keep only alphanumeric characters
    cleaned = ''.join(ch.lower() for ch in text if ch.isalnum())
    return cleaned == cleaned[::-1]
import unittest
from palindrome import is_palindrome

class TestIsPalindrome(unittest.TestCase):
    def test_empty_string(self):
        self.assertTrue(is_palindrome(''))

    def test_only_spaces(self):
        self.assertTrue(is_palindrome('   '))

    def test_only_punctuation(self):
        self.assertTrue(is_palindrome('!@#$%'))

    def test_simple_palindrome(self):
        self.assertTrue(is_palindrome('Racecar'))

    def test_palindrome_with_spaces_and_punctuation(self):
        self.assertTrue(is_palindrome('A man, a plan, a canal: Panama'))

    def test_non_palindrome(self):
        self.assertFalse(is_palindrome('Hello, world!'))

    def test_mixed_case_and_spaces(self):
        self.assertTrue(is_palindrome('No ' 'on'))

    def test_numbers(self):
        self.asser

### Проверка 2.2

In [ ]:
from src.agents import AgentConfigRegistry, ResearchAgent
from src.llm import LLMClient
from src.tools import SearchTool

research_llm_client = LLMClient(timeout=60.0)
research_search_tool = SearchTool(timeout=20.0)

research_agent = ResearchAgent(
    client=research_llm_client,
    search_tool=research_search_tool,
    config=AgentConfigRegistry().get_agent("research"),
    agent_id="research-notebook",
)

def show_research_report(report):
    print("Тема:", report.topic)
    print("Поисковые запросы:", report.search_queries)
    print("\nРезюме:\n", report.summary)

    print("\nКлючевые выводы:")
    for finding in report.key_findings:
        print("-", finding.claim)
        print("  Источники:", ", ".join(finding.source_urls) or "не указаны")

    print("\nИсточники:")
    for source in report.sources:
        print(f"- {source.title} — {source.url} (релевантность: {source.relevance_score})")
        print(" ", source.snippet[:400])

    print("\nОграничения:")
    for limitation in report.limitations:
        print("-", limitation)

In [ ]:
rag_report = research_agent.run_research(
    "Что такое Retrieval-Augmented Generation и как он работает?"
)
show_research_report(rag_report)

Тема: Что такое Retrieval-Augmented Generation и как он работает?
Поисковые запросы: ['Retrieval-Augmented Generation definition overview', 'how RAG works architecture retriever generator', 'RAG vs fine-tuning comparison advantages', 'Retrieval-Augmented Generation use cases examples']

Резюме:
 Retrieval-Augmented Generation (RAG) — это техника усиления больших языковых моделей (LLM) за счёт подключения внешних источников знаний. Система RAG состоит из базы знаний, ретривера (поиска), генератора (LLM) и часто интеграционного слоя. Процесс делится на два этапа: ingestion (подготовка данных: чанкинг, эмбеддинги, векторная БД) и inference (преобразование запроса в вектор, поиск релевантных документов, дополнение промпта найденным контекстом, генерация ответа). RAG позволяет использовать актуальные и доменно-специфичные данные без переобучения модели, снижает галлюцинации и обеспечивает прозрачность источников. В отличие от fine-tuning, RAG поддерживает динамическое обновление знаний и не

In [ ]:
comparison_report = research_agent.run_research(
    "Сравни RAG и fine-tuning для корпоративных чат-ботов. "
    "Рассмотри преимущества, недостатки, стоимость и актуальность информации."
)
show_research_report(comparison_report)

Тема: Сравни RAG и fine-tuning для корпоративных чат-ботов. Рассмотри преимущества, недостатки, стоимость и актуальность информации.
Поисковые запросы: ['RAG vs fine-tuning corporate chatbots advantages disadvantages', 'cost comparison RAG fine-tuning enterprise LLM deployment', 'information freshness accuracy RAG vs fine-tuning knowledge base updates', 'when to use RAG vs fine-tuning for internal company chatbot']

Резюме:
 RAG и fine‑tuning решают разные задачи: RAG подключает актуальные внешние знания во время инференса, а fine‑tuning закрепляет доменную экспертизу в весах модели. Для корпоративных чат‑ботов выбор зависит от частоты обновления знаний, бюджета, требований к задержке, трассируемости и управлению доступом. Гибридные подходы (RAG + fine‑tuning) часто используются в продакшене для сочетания свежести фактов с качеством форматирования и тона.

Ключевые выводы:
- RAG предназначен для внедрения новых знаний (факты, цены, политики), fine‑tuning — для изменения поведения модел

In [ ]:
context_report = research_agent.run_research(
    "Какие ограничения у современных LLM при работе с длинным контекстом?"
)
show_research_report(context_report)

Тема: Какие ограничения у современных LLM при работе с длинным контекстом?
Поисковые запросы: ['LLM long context window limitations performance degradation', 'transformer attention mechanism quadratic complexity long sequences', 'evaluation benchmarks long context LLMs needle in a haystack', 'techniques extend context length LLMs RoPE YaRN sliding window']

Резюме:
 Современные LLM сталкиваются с фундаментальными ограничениями при обработке длинного контекста. Основная проблема — деградация качества (context window degradation) по мере увеличения длины входа, которая проявляется задолго до заявленного лимита контекстного окна. Ключевые факторы: квадратичная сложность механизма внимания (O(n²)), феномен «lost in the middle» (ухудшение извлечения информации из середины контекста), падение производительности даже при идеальном ретриве и эмпирические практические лимиты, значительно ниже рекламных. Существуют архитектурные предложения (линейное внимание, разреженное внимание, CPA) для прео

In [ ]:
import json
from unittest.mock import Mock

from src.agents import AgentConfigRegistry, ResearchAgent
from src.llm import LLMClient, LLMResponse
from src.tools import SearchTool

mock_llm = Mock(spec=LLMClient)
mock_llm.generate.return_value = LLMResponse(
    content=json.dumps({"search_queries": ["тестовый запрос"]}),
    model="mock-model",
    prompt_tokens=10,
    completion_tokens=10,
    total_tokens=20,
    finish_reason="stop",
)

mock_search = Mock(spec=SearchTool)
mock_search.search_many.return_value = []

empty_result_agent = ResearchAgent(
    client=mock_llm,
    search_tool=mock_search,
    config=AgentConfigRegistry().get_agent("research"),
    agent_id="research-empty-results-test",
)

empty_report = empty_result_agent.run_research("Проверка пустой поисковой выдачи")
show_research_report(empty_report)

assert empty_report.sources == []
assert empty_report.key_findings == []
mock_search.search_many.assert_called_once()
mock_llm.generate.assert_called_once()  # Только планирование; синтез при пустой выдаче не вызывается.

Тема: Проверка пустой поисковой выдачи
Поисковые запросы: ['тестовый запрос']

Резюме:
 Поисковый сервис не вернул материалов, достаточных для исследования.

Ключевые выводы:

Источники:

Ограничения:
- По указанным запросам не найдено пригодных источников.


### 2.3

In [ ]:
from src.agents import AgentConfigRegistry, MathAgent
from src.llm import LLMClient

math_llm_client = LLMClient(timeout=90.0)
math_agent = MathAgent(
    client=math_llm_client,
    config=AgentConfigRegistry().get_agent("math"),
    agent_id="math-notebook",
)

def reset_math_agent():
    """Create a fresh client so cells also work when run out of order."""
    global math_llm_client, math_agent
    previous_client = globals().get("math_llm_client")
    if previous_client is not None:
        previous_client.close()
    math_llm_client = LLMClient(timeout=90.0)
    math_agent = MathAgent(
        client=math_llm_client,
        config=AgentConfigRegistry().get_agent("math"),
        agent_id="math-notebook",
    )

def show_math_report(report):
    print("Задача:", report.problem)
    print("Формализация:", report.mathematical_formulation)
    print("Метод:", report.method)
    print("\nВычисления:")
    for item in report.calculations:
        print("-", item)
    print("\nРезультат:", report.result)
    print("\nПроверка:", report.verification)
    print("Проверено:", report.verified)
    print("\nОбъяснение:", report.explanation)

    if report.assumptions:
        print("\nПредположения:", report.assumptions)
    if report.needs_clarification:
        print("\nНужно уточнить:", report.clarification_question)

In [ ]:
reset_math_agent()
report = math_agent.solve(
    "Найди первую и вторую производные функции f(x) = x**3 + 2*x**2 - 5*x."
)
show_math_report(report)

Задача: Найди первую и вторую производные функции f(x) = x**3 + 2*x**2 - 5*x.
Формализация: Найти первую и вторую производные функции f(x) = x^3 + 2x^2 - 5x по переменной x.
Метод: Символьное дифференцирование

Вычисления:
- derivative: {"derivative": "3*x**2 + 4*x - 5", "order": 1, "variable": "x"}
- derivative: {"derivative": "2*(3*x + 2)", "order": 2, "variable": "x"}

Результат: [{"operation": "derivative", "expression": "x**3 + 2*x**2 - 5*x", "result": {"variable": "x", "order": 1, "derivative": "3*x**2 + 4*x - 5"}, "verified": true, "verification": "Finite-difference checks agreed with the symbolic derivative.", "diagnostics": {"finite_difference_errors": [1.00000039804371e-06, 9.999993171305732e-07]}}, {"operation": "derivative", "expression": "x**3 + 2*x**2 - 5*x", "result": {"variable": "x", "order": 2, "derivative": "2*(3*x + 2)"}, "verified": true, "verification": "Finite-difference checks agreed with the symbolic derivative.", "diagnostics": {"finite_difference_errors": [5.

In [ ]:
reset_math_agent()
report = math_agent.solve(
    "Найди численный локальный минимум функции f(x) = x**2 - 6*x + 11. "
    "Используй начальное приближение x=0."
)
show_math_report(report)

Задача: Найди численный локальный минимум функции f(x) = x**2 - 6*x + 11. Используй начальное приближение x=0.
Формализация: Найти локальный минимум функции f(x) = x^2 - 6x + 11, используя численный метод с начальным приближением x = 0.
Метод: Численная минимизация (например, метод BFGS или аналогичный градиентный метод)

Вычисления:
- minimize: {"bounds_satisfied": true, "constraint_values": [], "constraints_satisfied": true, "global_optimum_guaranteed": false, "iterations": 3, "method": "BFGS", "objective_value": 2.0, "optimizer_success": true, "status": 0, "variables": {"x": 2.999999970492759}}

Результат: [{"operation": "minimize", "expression": "x**2 - 6*x + 11", "result": {"variables": {"x": 2.999999970492759}, "objective_value": 2.0, "optimizer_success": true, "status": 0, "iterations": 3, "method": "BFGS", "bounds_satisfied": true, "constraints_satisfied": true, "constraint_values": [], "global_optimum_guaranteed": false}, "verified": true, "verification": "Optimizer status is 

In [ ]:
reset_math_agent()
report = math_agent.solve(
    "Найди численный минимум функции "
    "f(x, y) = (x - 2)**2 + (y + 3)**2 "
    "с начальным приближением x=0, y=0."
)
show_math_report(report)

Задача: Найди численный минимум функции f(x, y) = (x - 2)**2 + (y + 3)**2 с начальным приближением x=0, y=0.
Формализация: Минимизировать f(x, y) = (x - 2)^2 + (y + 3)^2 относительно x, y ∈ ℝ без ограничений.
Метод: Численная безусловная минимизация (например, BFGS или Nelder-Mead).

Вычисления:
- minimize: {"bounds_satisfied": true, "constraint_values": [], "constraints_satisfied": true, "global_optimum_guaranteed": false, "iterations": 4, "method": "BFGS", "objective_value": 1.1487639851209958e-16, "optimizer_success": true, "status": 0, "variables": {"x": 1.9999999929913097, "y": -3.0000000081089246}}

Результат: [{"operation": "minimize", "expression": "(x - 2)**2 + (y + 3)**2", "result": {"variables": {"x": 1.9999999929913097, "y": -3.0000000081089246}, "objective_value": 1.1487639851209958e-16, "optimizer_success": true, "status": 0, "iterations": 4, "method": "BFGS", "bounds_satisfied": true, "constraints_satisfied": true, "constraint_values": [], "global_optimum_guaranteed": fa

In [ ]:
reset_math_agent()
report = math_agent.solve(
    "Минимизируй x**2 + y**2 при ограничениях x + y = 2, "
    "x >= 0, y >= 0. Используй начальное приближение x=1, y=1."
)
show_math_report(report)

Задача: Минимизируй x**2 + y**2 при ограничениях x + y = 2, x >= 0, y >= 0. Используй начальное приближение x=1, y=1.
Формализация: Minimize f(x, y) = x^2 + y^2 subject to equality constraint x + y = 2 and inequality constraints x >= 0, y >= 0.
Метод: Sequential Least Squares Programming (SLSQP) for constrained nonlinear optimization

Вычисления:
- minimize: {"bounds_satisfied": true, "constraint_values": [0.0], "constraints_satisfied": true, "global_optimum_guaranteed": false, "iterations": 1, "method": "SLSQP", "objective_value": 2.0, "optimizer_success": true, "status": 0, "variables": {"x": 1.0, "y": 1.0}}

Результат: [{"operation": "minimize", "expression": "x**2 + y**2", "result": {"variables": {"x": 1.0, "y": 1.0}, "objective_value": 2.0, "optimizer_success": true, "status": 0, "iterations": 1, "method": "SLSQP", "bounds_satisfied": true, "constraints_satisfied": true, "constraint_values": [0.0], "global_optimum_guaranteed": false}, "verified": true, "verification": "Optimizer s

In [ ]:
reset_math_agent()
report = math_agent.solve("Найди минимум.")
show_math_report(report)

Задача: Найди минимум.
Формализация: Задача не полностью определена: не указана целевая функция, переменные, область определения или ограничения.
Метод: Не выбран — требуется уточнение условия.

Вычисления:

Результат: Решение не вычислялось: требуется уточнение условия.

Проверка: Проверка не выполнялась, поскольку MathTool не запускался.
Проверено: False

Объяснение: Укажите, минимум какой функции (целевой функции) и по каким переменным нужно найти. Приведите явное выражение функции, имена переменных и, если есть, ограничения (равенства, неравенства или границы переменных).

Нужно уточнить: Укажите, минимум какой функции (целевой функции) и по каким переменным нужно найти. Приведите явное выражение функции, имена переменных и, если есть, ограничения (равенства, неравенства или границы переменных).


In [ ]:
math_llm_client.close()

### 2.4

In [2]:
from src.agents import AgentConfigRegistry, CriticAgent, CriticAgentError
from src.llm import LLMClient
from src.tools import CriticTool

critic_tool = CriticTool()

def reset_critic_agent():
    global critic_client, critic_agent

    previous_client = globals().get("critic_client")
    if previous_client is not None:
        previous_client.close()

    critic_client = LLMClient(timeout=90.0)
    critic_agent = CriticAgent(
        client=critic_client,
        config=AgentConfigRegistry().get_agent("critic"),
        agent_id="critic-notebook",
    )

def show_critic_review(task, reviewed_result, report):
    print("Исходная задача:\n", task)
    print("\nПроверяемый результат:\n", reviewed_result)
    print("\nЗамечания:")
    if report.issues:
        for issue in report.issues:
            print(f"- [{issue.severity}/{issue.category}] {issue.description}")
            print("  Рекомендация:", issue.recommendation)
    else:
        print("Нет выявленных замечаний.")

    print("\nОценка:", report.score)
    print("Вердикт:", report.verdict)
    print("Обоснование:", report.summary)
    print("Программные проверки:", report.checks)
    print("Ограничения:", report.verification_limits)

def run_critic_review(
    task,
    result,
    result_type,
    agent_name,
    additional_materials=None,
):
    reset_critic_agent()

    try:
        report = critic_agent.review(
            task,
            result,
            result_type=result_type,
            agent_name=agent_name,
            additional_materials=additional_materials,
        )
    except CriticAgentError as error:
        print(f"LLM-вердикт не получен: {type(error).__name__}: {error}")
        print("Показываю только локальные проверки; общего вердикта нет.")

        if result_type == "coding":
            checks = [
                critic_tool.check_structure(
                    result,
                    required_elements=("code", "tests"),
                ),
                critic_tool.check_python(result.get("code", ""), task=task),
            ]
        elif result_type == "research":
            sources = (additional_materials or {}).get("sources", [])
            checks = [
                critic_tool.check_source_urls(str(result), sources)
            ]
        elif result_type == "math":
            checks = [
                critic_tool.check_math_value(
                    expression=result["expression"],
                    variable=result.get("variable", "x"),
                    reported_point=float(
                        result.get("reported_point", result.get("point"))
                    ),
                    reported_value=float(
                        result.get("reported_value", result.get("value"))
                    ),
                )
            ]
        else:
            checks = []

        for check in checks:
            print("Проверки:", check.checks)
            for finding in check.findings:
                print(
                    f"Замечание: [{finding.severity}/{finding.category}] "
                    f"{finding.description}"
                )
            if check.limitations:
                print("Ограничения:", check.limitations)

        return None

    show_critic_review(task, result, report)
    return report

In [3]:
task_critic_1 = (
    "Реализовать бинарный поиск в отсортированном списке. "
    "Вернуть индекс найденного элемента или -1, если его нет. Добавить тесты."
)

result_critic_1 = {
    "code": """def binary_search(items, target):
    left, right = 0, len(items) - 1
    while left <= right:
        mid = (left + right) // 2
        if items[mid] == target:
            return mid
        if items[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return -1
""",
    "tests": "Проверить найденное значение, пустой список и отсутствующий элемент.",
}

report_critic_1 = run_critic_review(
    task_critic_1, result_critic_1, "coding", "Coding Agent"
)


LLM-вердикт не получен: CriticLLMError: Critic review call failed: RateLimitError: OpenRouter rate limit was exceeded.
Показываю только локальные проверки; общего вердикта нет.
Проверки: [CriticCheck(name='result_structure', status='passed', details='All requested fields are present.')]
Проверки: [CriticCheck(name='python_ast', status='passed', details='Python source parsed successfully.'), CriticCheck(name='dangerous_dynamic_execution', status='passed', details='No eval/exec/compile/__import__ call found.')]


In [4]:
task_critic_2 = (
    "Реализовать бинарный поиск в отсортированном списке. "
    "Вернуть индекс найденного элемента или -1, если его нет."
)

result_critic_2 = {
    "code": """def binary_search(items, target):
    left, right = 0, len(items) - 1
    while left <= right:
        mid = (left + right) // 2
        if items[mid] == target:
            return mid
        if items[mid] < target:
            left = mid + 1
        else:
            right = mid - 1
    return 0
""",
    "tests": "Проверить найденный и отсутствующий элемент.",
}

report_critic_2 = run_critic_review(
    task_critic_2, result_critic_2, "coding", "Coding Agent"
)

LLM-вердикт не получен: CriticLLMError: Critic review call failed: RateLimitError: OpenRouter rate limit was exceeded.
Показываю только локальные проверки; общего вердикта нет.
Проверки: [CriticCheck(name='result_structure', status='passed', details='All requested fields are present.')]
Проверки: [CriticCheck(name='python_ast', status='passed', details='Python source parsed successfully.'), CriticCheck(name='dangerous_dynamic_execution', status='passed', details='No eval/exec/compile/__import__ call found.')]


In [5]:
task_critic_3 = (
    "Кратко сравнить RAG и fine-tuning и привести подтверждённые выводы с источниками."
)

sources_critic_3 = [
    {
        "url": "https://arxiv.org/abs/2005.11401",
        "title": "Retrieval-Augmented Generation for Knowledge-Intensive NLP Tasks",
        "snippet": (
            "В работе RAG объединяет языковую модель с непараметрической памятью, "
            "из которой извлекаются документы."
        ),
    }
]

result_critic_3 = (
    "RAG дополняет генерацию извлечёнными внешними документами и позволяет "
    "использовать обновляемую базу знаний. Это отличается от fine-tuning, "
    "при котором модель адаптируется на обучающих примерах. "
    "Источник по устройству RAG: https://arxiv.org/abs/2005.11401"
)

report_critic_3 = run_critic_review(
    task_critic_3,
    result_critic_3,
    "research",
    "Research Agent",
    additional_materials={"sources": sources_critic_3},
)


LLM-вердикт не получен: CriticLLMError: Critic review call failed: RateLimitError: OpenRouter rate limit was exceeded.
Показываю только локальные проверки; общего вердикта нет.
Проверки: [CriticCheck(name='source_url_match', status='passed', details='Every cited URL matches a provided source URL.')]


In [6]:
task_critic_4 = "Сравнить RAG и fine-tuning, используя только предоставленные материалы."

sources_critic_4 = [
    {
        "url": "https://arxiv.org/abs/2005.11401",
        "snippet": "RAG использует извлечение документов при генерации.",
    }
]

result_critic_4 = "RAG всегда точнее fine-tuning (https://unsupported.example/paper)."

report_critic_4 = run_critic_review(
    task_critic_4,
    result_critic_4,
    "research",
    "Research Agent",
    additional_materials={"sources": sources_critic_4},
)


LLM-вердикт не получен: CriticLLMError: Critic review call failed: RateLimitError: OpenRouter rate limit was exceeded.
Показываю только локальные проверки; общего вердикта нет.
Проверки: [CriticCheck(name='source_url_match', status='failed', details='Unmatched URLs: https://unsupported.example/paper')]
Замечание: [MAJOR/EVIDENCE] Cited URL 'https://unsupported.example/paper' is not present in the provided source list.


In [ ]:
task_critic_5 = "Найти минимум f(x) = x**2 - 6*x + 11."

result_critic_5 = {
    "expression": "x**2 - 6*x + 11",
    "variable": "x",
    "reported_point": 3,
    "reported_value": 2,
    "method": "Исследование квадратичной функции.",
}

report_critic_5 = run_critic_review(
    task_critic_5, result_critic_5, "math", "Math Agent"
)


LLM-вердикт не получен: CriticLLMError: Critic review call failed: RateLimitError: OpenRouter rate limit was exceeded.
Показываю только локальные проверки; общего вердикта нет.
Проверки: [CriticCheck(name='math_substitution', status='passed', details='Recomputed f(x=3.0)=2.0; derivative=0.0.')]


In [ ]:
task_critic_6 = "Найти минимум f(x) = x**2 - 6*x + 11."

result_critic_6 = {
    "expression": "x**2 - 6*x + 11",
    "variable": "x",
    "reported_point": 2,
    "reported_value": 3,
    "method": "Численный поиск минимума.",
}

report_critic_6 = run_critic_review(
    task_critic_6, result_critic_6, "math", "Math Agent"
)


LLM-вердикт не получен: CriticLLMError: Critic review call failed: RateLimitError: OpenRouter rate limit was exceeded.
Показываю только локальные проверки; общего вердикта нет.
Проверки: [CriticCheck(name='math_substitution', status='failed', details='Recomputed f(x=2.0)=3.0; derivative=-2.0.')]
Замечание: [MAJOR/CORRECTNESS] The derivative at the reported minimum point is -2.0, not approximately zero.
Ограничения: ['This local check verifies the reported value and first-order stationarity only; it does not establish global optimality.']


### 2.5

In [7]:
from src.agents import (
    AgentConfigRegistry,
    CodingAgent,
    CriticAgent,
    MathAgent,
    Orchestrator,
    OrchestratorPlan,
    ResearchAgent,
    Subtask,
)
from src.llm import LLMClient
from src.tools import SearchTool

agent_configs = AgentConfigRegistry()

def reset_orchestrator_agents():
    global orchestrator_client, research_agent, math_agent, coding_agent
    global critic_agent_for_orchestrator, orchestrator
    previous_client = globals().get("orchestrator_client")
    if previous_client is not None:
        previous_client.close()
    orchestrator_client = LLMClient(timeout=90.0)
    research_agent = ResearchAgent(
        client=orchestrator_client,
        search_tool=SearchTool(),
        config=agent_configs.get_agent("research"),
        agent_id="research-via-orchestrator",
    )
    math_agent = MathAgent(
        client=orchestrator_client,
        config=agent_configs.get_agent("math"),
        agent_id="math-via-orchestrator",
    )
    coding_agent = CodingAgent(
        client=orchestrator_client,
        config=agent_configs.get_agent("coding"),
        agent_id="coding-via-orchestrator",
    )
    critic_agent_for_orchestrator = CriticAgent(
        client=orchestrator_client,
        config=agent_configs.get_agent("critic"),
        agent_id="critic-via-orchestrator",
    )
    orchestrator = Orchestrator(
        client=orchestrator_client,
        agents={
            "research": research_agent,
            "math": math_agent,
            "coding": coding_agent,
            "critic": critic_agent_for_orchestrator,
        },
        config=agent_configs.get_agent("orchestrator"),
    )

reset_orchestrator_agents()

def show_orchestrator_result(result):
    print("Исходный запрос:\n", result.original_request)
    print("\nПлан:")
    if result.plan is None:
        print("План не сформирован.")
    else:
        for item in result.plan.subtasks:
            print(
                f"- {item.id}: agent={item.agent}; "
                f"depends_on={item.depends_on}; status={item.status}"
            )
            print("  Задача:", item.description)
            if item.result is not None:
                print("  Результат:", item.result)
            if item.error:
                print("  Ошибка:", item.error)
        print("Порядок выполнения:", result.execution_order)

    print("\nОшибки:", result.errors)
    print("\nИтоговый ответ:\n", result.final_answer)
    print("\nСтатус:", result.status)
    if any("RateLimitError" in error for error in result.errors):
        print("Провайдер ограничил частоту запросов. Повторите позже; это не результат работы агентов.")


In [8]:
reset_orchestrator_agents()
result_orchestrator_1 = orchestrator.run(
    "Напиши функцию бинарного поиска на Python."
)
show_orchestrator_result(result_orchestrator_1)

Исходный запрос:
 Напиши функцию бинарного поиска на Python.

План:
План не сформирован.

Ошибки: ['Planning failed: OrchestratorLLMError: Could not plan the request: RateLimitError: OpenRouter rate limit was exceeded.']

Итоговый ответ:
 Planning failed: OrchestratorLLMError: Could not plan the request: RateLimitError: OpenRouter rate limit was exceeded.

Статус: FAILED
Провайдер ограничил частоту запросов. Повторите позже; это не результат работы агентов.


In [9]:
reset_orchestrator_agents()
result_orchestrator_2 = orchestrator.run(
    "Исследуй, что такое Retrieval-Augmented Generation, "
    "и объясни его преимущества и недостатки."
)
show_orchestrator_result(result_orchestrator_2)

Исходный запрос:
 Исследуй, что такое Retrieval-Augmented Generation, и объясни его преимущества и недостатки.

План:
План не сформирован.

Ошибки: ['Planning failed: OrchestratorLLMError: Could not plan the request: RateLimitError: OpenRouter rate limit was exceeded.']

Итоговый ответ:
 Planning failed: OrchestratorLLMError: Could not plan the request: RateLimitError: OpenRouter rate limit was exceeded.

Статус: FAILED
Провайдер ограничил частоту запросов. Повторите позже; это не результат работы агентов.


In [10]:
reset_orchestrator_agents()
result_orchestrator_3 = orchestrator.run(
    "Исследуй алгоритмы оптимизации маршрутов доставки, выбери подходящий, "
    "формализуй задачу, реализуй алгоритм на Python и проверь результат."
)
show_orchestrator_result(result_orchestrator_3)

Исходный запрос:
 Исследуй алгоритмы оптимизации маршрутов доставки, выбери подходящий, формализуй задачу, реализуй алгоритм на Python и проверь результат.

План:
План не сформирован.

Ошибки: ['Planning failed: OrchestratorLLMError: Could not plan the request: RateLimitError: OpenRouter rate limit was exceeded.']

Итоговый ответ:
 Planning failed: OrchestratorLLMError: Could not plan the request: RateLimitError: OpenRouter rate limit was exceeded.

Статус: FAILED
Провайдер ограничил частоту запросов. Повторите позже; это не результат работы агентов.


In [ ]:
reset_orchestrator_agents()
from src.agents import OrchestratorLLMError, OrchestratorResult

class FailingMathAgent:
    def solve(self, task):
        raise RuntimeError("Преднамеренная ошибка для проверки Orchestrator")

failure_orchestrator = Orchestrator(
    client=orchestrator_client,
    agents={
        "research": research_agent,
        "math": FailingMathAgent(),
        "coding": coding_agent,
    },
    config=agent_configs.get_agent("orchestrator"),
)

failure_plan = OrchestratorPlan(
    original_request="Исследовать задачу, формализовать её и реализовать решение.",
    subtasks=[
        Subtask(
            "research",
            "Исследуй алгоритм оптимизации маршрута доставки.",
            "research",
            [],
        ),
        Subtask(
            "math",
            "Формализуй математическую задачу оптимизации.",
            "math",
            ["research"],
        ),
        Subtask(
            "coding",
            "Реализуй постановку на Python.",
            "coding",
            ["research", "math"],
        ),
    ],
)

failure_orchestrator.validate_plan(failure_plan)
failure_orchestrator.execute_plan(failure_plan)
try:
    failure_answer = failure_orchestrator.build_final_answer(
        failure_plan.original_request,
        failure_plan,
    )
except OrchestratorLLMError as error:
    failure_answer = (
        "Итоговый LLM-синтез недоступен: "
        f"{type(error).__name__}: {error}. Статусы подзадач показаны отдельно."
    )

failure_status = (
    "PARTIAL"
    if any(task.status == "COMPLETED" for task in failure_plan.subtasks)
    else "FAILED"
)
failure_result = OrchestratorResult(
    original_request=failure_plan.original_request,
    plan=failure_plan,
    subtask_results={task.id: task.result for task in failure_plan.subtasks},
    final_answer=failure_answer,
    status=failure_status,
    errors=[
        *[f"{task.id}: {task.error}" for task in failure_plan.subtasks if task.error],
        *([failure_answer] if "Итоговый LLM-синтез недоступен" in failure_answer else []),
    ],
    execution_order=list(failure_plan.execution_order),
)

show_orchestrator_result(failure_result)

Исходный запрос:
 Исследовать задачу, формализовать её и реализовать решение.

План:
- research: agent=research; depends_on=[]; status=FAILED
  Задача: Исследуй алгоритм оптимизации маршрута доставки.
  Ошибка: ResearchLLMError: Could not plan research searches: RateLimitError: OpenRouter rate limit was exceeded.
- math: agent=math; depends_on=['research']; status=SKIPPED
  Задача: Формализуй математическую задачу оптимизации.
  Ошибка: Skipped because dependency/dependencies failed: research
- coding: agent=coding; depends_on=['research', 'math']; status=SKIPPED
  Задача: Реализуй постановку на Python.
  Ошибка: Skipped because dependency/dependencies failed: research, math
Порядок выполнения: ['research', 'math', 'coding']

Ошибки: ['research: ResearchLLMError: Could not plan research searches: RateLimitError: OpenRouter rate limit was exceeded.', 'math: Skipped because dependency/dependencies failed: research', 'coding: Skipped because dependency/dependencies failed: research, math'

In [ ]:
from src.agents import OrchestratorLLMError, OrchestratorResult

reset_orchestrator_agents()

class FailingMathAgent:
    def solve(self, task):
        raise RuntimeError("Преднамеренная ошибка для проверки Orchestrator")

failure_orchestrator = Orchestrator(
    client=orchestrator_client,
    agents={
        "research": research_agent,
        "math": FailingMathAgent(),
        "coding": coding_agent,
    },
    config=agent_configs.get_agent("orchestrator"),
)

failure_plan = OrchestratorPlan(
    original_request="Исследовать задачу, формализовать её и реализовать решение.",
    subtasks=[
        Subtask(
            "research",
            "Исследуй алгоритм оптимизации маршрута доставки.",
            "research",
            [],
        ),
        Subtask(
            "math",
            "Формализуй математическую задачу оптимизации.",
            "math",
            ["research"],
        ),
        Subtask(
            "coding",
            "Реализуй постановку на Python.",
            "coding",
            ["research", "math"],
        ),
    ],
)

failure_orchestrator.validate_plan(failure_plan)
failure_orchestrator.execute_plan(failure_plan)

try:
    failure_answer = failure_orchestrator.build_final_answer(
        failure_plan.original_request,
        failure_plan,
    )
except OrchestratorLLMError as error:
    failure_answer = (
        "Итоговый LLM-синтез недоступен: "
        f"{type(error).__name__}: {error}. Статусы подзадач показаны ниже."
    )

failure_status = (
    "PARTIAL"
    if any(item.status == "COMPLETED" for item in failure_plan.subtasks)
    else "FAILED"
)

failure_result = OrchestratorResult(
    original_request=failure_plan.original_request,
    plan=failure_plan,
    subtask_results={
        item.id: item.result for item in failure_plan.subtasks
    },
    final_answer=failure_answer,
    status=failure_status,
    errors=[
        f"{item.id}: {item.error}"
        for item in failure_plan.subtasks
        if item.error
    ],
    execution_order=list(failure_plan.execution_order),
)

show_orchestrator_result(failure_result)

Исходный запрос:
 Исследовать задачу, формализовать её и реализовать решение.

План:
- research: agent=research; depends_on=[]; status=FAILED
  Задача: Исследуй алгоритм оптимизации маршрута доставки.
  Ошибка: ResearchLLMError: Could not plan research searches: RateLimitError: OpenRouter rate limit was exceeded.
- math: agent=math; depends_on=['research']; status=SKIPPED
  Задача: Формализуй математическую задачу оптимизации.
  Ошибка: Skipped because dependency/dependencies failed: research
- coding: agent=coding; depends_on=['research', 'math']; status=SKIPPED
  Задача: Реализуй постановку на Python.
  Ошибка: Skipped because dependency/dependencies failed: research, math
Порядок выполнения: ['research', 'math', 'coding']

Ошибки: ['research: ResearchLLMError: Could not plan research searches: RateLimitError: OpenRouter rate limit was exceeded.', 'math: Skipped because dependency/dependencies failed: research', 'coding: Skipped because dependency/dependencies failed: research, math'

In [ ]:
critic_client.close()
orchestrator_client.close()

### 3.1